In [1]:
from src.preferential_attachment_graph import PreferentialAttachmentGraph, FitnessType
from src.drawer import Drawer

In [2]:
G = PreferentialAttachmentGraph(n=50, m=2, fitness=FitnessType.POLY, fitness_alpha=1, track_history=False, select_planar_subgraph=True, test_planarity_restoring=True)
G.restoring_test_result[:5]

Planarity restoring test produced 1140 result files.


[{'added_edge': {'source': 0, 'target': 5},
  'min_edges_to_remove': 0,
  'removal_options': [[]]},
 {'added_edge': {'source': 0, 'target': 8},
  'min_edges_to_remove': 0,
  'removal_options': [[]]},
 {'added_edge': {'source': 0, 'target': 23},
  'min_edges_to_remove': 0,
  'removal_options': [[]]},
 {'added_edge': {'source': 2, 'target': 46},
  'min_edges_to_remove': 1,
  'removal_options': [[{'source': 19, 'target': 38}],
   [{'source': 39, 'target': 48}],
   [{'source': 39, 'target': 46}],
   [{'source': 13, 'target': 19}],
   [{'source': 0, 'target': 13}],
   [{'source': 23, 'target': 38}],
   [{'source': 23, 'target': 48}],
   [{'source': 2, 'target': 19}]]},
 {'added_edge': {'source': 32, 'target': 44},
  'min_edges_to_remove': 0,
  'removal_options': [[]]}]

In [5]:
N_VALUES = [100]
M_VALUES = [2, 3]
FITNESS_FUNCTIONS = [
    {"type": FitnessType.LINEAR, "alpha": 1.0},
    {"type": FitnessType.POLY, "alpha": 0.2},
    {"type": FitnessType.POLY, "alpha": 0.5},
    {"type": FitnessType.POLY, "alpha": 0.8},
    {"type": FitnessType.LOG, "alpha": 1},
]
REPS = 1
RESULTS_DIR = "results/planarity_restoring"

In [6]:
import os
import json
import itertools
import networkx as nx
from concurrent.futures import ProcessPoolExecutor, as_completed

import planarity_restoring_worker as worker

os.makedirs(RESULTS_DIR, exist_ok=True)


def get_result_filepath(params: dict) -> str:
    """Get the filepath for a given parameter combination."""
    filename = (
        f"n{params['n']}_m{params['m']}_{params['fitness_type'].name}_a{params['fitness_alpha']}_rep{params['rep']}_restoring.json"
    )
    return os.path.join(RESULTS_DIR, filename)


def is_already_done(params: dict) -> bool:
    """Check if a result file already exists and is valid."""
    filepath = get_result_filepath(params)
    if not os.path.exists(filepath):
        return False
    try:
        with open(filepath, "r") as f:
            data = json.load(f)
        return "restoring_results" in data and "summary" in data and "params" in data
    except (json.JSONDecodeError, IOError):
        return False


def generate_single(params: dict) -> dict:
    return worker.generate_single(params, RESULTS_DIR)

all_tasks = []
for n, m, fitness, rep in itertools.product(
    N_VALUES, M_VALUES, FITNESS_FUNCTIONS, range(REPS)
):
    all_tasks.append({
        "n": n,
        "m": m,
        "fitness_type": fitness["type"],
        "fitness_alpha": fitness["alpha"],
        "rep": rep,
    })

tasks = [t for t in all_tasks if not is_already_done(t)]
already_done = len(all_tasks) - len(tasks)

print(f"Total tasks: {len(all_tasks)} | Already done: {already_done} | Remaining: {len(tasks)}")

if not tasks:
    print("All planarity-restoring tasks already completed.")
else:
    all_results = []
    num_workers = min(6, os.cpu_count() or 1)

    print(f"Running with ProcessPoolExecutor ({num_workers} workers)...\n")
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(worker.generate_single, task, RESULTS_DIR): task for task in tasks}
        for i, future in enumerate(as_completed(futures), 1):
            task = futures[future]
            try:
                result = future.result()
                all_results.append(result)
                summary = result["summary"]
                print(
                    f"[{i}/{len(tasks)}] n={task['n']} m={task['m']} {task['fitness_type'].name}(α={task['fitness_alpha']}) rep{task['rep']}: "
                    f"edges={summary['num_edges']}, restoring_cases={summary['num_restoring_cases']}"
                )
            except Exception as e:
                print(f"[{i}/{len(tasks)}] FAILED: {task} -> {e}")

    print(f"Done. Results saved to: {RESULTS_DIR}")

Total tasks: 10 | Already done: 0 | Remaining: 10
Running with ProcessPoolExecutor (6 workers)...



Process ForkServerProcess-9:
Traceback (most recent call last):
  File "/usr/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.14/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.14/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/usr/lib/python3.14/multiprocessing/queues.py", line 101, in get
    res = self._recv_bytes()
  File "/usr/lib/python3.14/multiprocessing/connection.py", line 226, in recv_bytes
    buf = self._recv_bytes(maxlength)
  File "/usr/lib/python3.14/multiprocessing/connection.py", line 451, in _recv_bytes
    buf = self._recv(4)
  File "/usr/lib/python3.14/multiprocessing/connection.py", line 416, in _recv
    chunk = read(handle, to_read)
KeyboardInterrupt


KeyboardInterrupt: 